In [1]:
import jiwer
from datasets import load_dataset, Audio

N_SAMPLES = 10

fleurs_id = load_dataset("google/fleurs", "id_id", split="validation", streaming=True)
fleurs_id = fleurs_id.cast_column("audio", Audio(decode=False))  # avoid torchcodec/torch dep, decode via soundfile instead
fleurs_id_samples = list(fleurs_id.take(N_SAMPLES))

print(f"loaded {len(fleurs_id_samples)} samples from fleurs id_id validation split")

loaded 10 samples from fleurs id_id validation split


In [2]:
from dataclasses import dataclass
from faster_whisper.transcribe import Segment

@dataclass
class TranscriptionResult:
    text: str
    language: str
    language_probability: float
    segments: list[Segment]
    audio_duration_s: float
    inference_time_s: float

    @property
    def realtime_factor(self) -> float:
        return self.audio_duration_s / self.inference_time_s

In [3]:
from faster_whisper import WhisperModel

device = "cpu"
compute_type = "float32"

# Audio config
SAMPLE_RATE = 16_000
CHANNELS = 1
DTYPE = "float32"

# Chunking config
CHUNK_DURATION_S = 3       # transcribe every N seconds
OVERLAP_DURATION_S = 0.5   # overlap between chunks to avoid cutting words mid-sentence
model = WhisperModel("large-v3-turbo", device=device, compute_type=compute_type)

In [4]:
import time

def transcribe(model: WhisperModel, audio_path: str, language: str | None = None, beam_size: int = 5) -> TranscriptionResult:
    """
    Transcribe an audio file.

    Args:
        model: Loaded WhisperModel instance.
        audio_path: Path to audio file (wav, mp3, m4a, etc.).
        language: ISO 639-1 code e.g. "id" for Indonesian, "en" for English.
                  None = auto-detect.
        beam_size: Beam search width. Higher = more accurate, slower.

    Returns:
        TranscriptionResult with text, segments, timing info.
    """

    t0 = time.perf_counter()
    
    segments_gen, info = model.transcribe(
        audio_path,
        language=language,
        beam_size=beam_size,
        vad_filter=True,
        vad_parameters=dict(
            min_silence_duration_ms=2000,  # wait 2s of silence before cutting
            speech_pad_ms=400,             # pad speech edges to avoid clipping
            threshold=0.3,                 # lower = more sensitive to speech
        ),
        word_timestamps=True
    )

    segments = list(segments_gen)
    full_text = " ".join(seg.text.strip() for seg in segments)
    elapsed = time.perf_counter() - t0

    return TranscriptionResult(
        text=full_text,
        language=info.language,
        language_probability=info.language_probability,
        segments=segments,
        audio_duration_s=info.duration,
        inference_time_s=elapsed
    )

In [5]:
import time
import queue
import tempfile
import threading
import soundfile as sf
import sounddevice as sd
import numpy as np
from collections import deque

def _audio_callback(indata: np.ndarray, frames: int, time_info, status, q: queue.Queue):
    if status:
        print(f"[audio] {status}")
    q.put(indata.copy().flatten())  # always 1D

def stream_transcribe(
    model: WhisperModel, 
    stop_event: threading.Event, 
    language: str | None = None, 
    chunk_s: float = CHUNK_DURATION_S, 
    overlap_s: float = OVERLAP_DURATION_S,
    output=None,
    device: int | None = None,
    debug_log: list | None = None,
):
    def emit(msg: str):
        if output is not None:
            with output:
                print(msg)
        else:
            print(msg)

    device_idx = device if device is not None else sd.default.device[0]
    device_name = sd.query_devices(device_idx)["name"]
    emit(f"[mic] using device #{device_idx}: {device_name}")

    audio_queue: queue.Queue[np.ndarray] = queue.Queue()
    chunk_samples = int(chunk_s * SAMPLE_RATE)
    overlap_samples = int(overlap_s * SAMPLE_RATE)

    buffer: deque[np.ndarray] = deque()
    buffer_len = 0
    start_time = time.time()
    chunk_idx = 0

    def transcription_loop():
        nonlocal buffer, buffer_len, chunk_idx

        while not stop_event.is_set():
            try:
                while True:
                    chunk = audio_queue.get_nowait()  # already 1D
                    buffer.append(chunk)
                    buffer_len += len(chunk)
            except queue.Empty:
                pass

            if buffer_len < chunk_samples:
                time.sleep(0.05)
                continue

            audio_np = np.concatenate(list(buffer))  # all 1D, safe

            overlap_audio = audio_np[-overlap_samples:] if overlap_samples > 0 else np.array([])
            buffer = deque([overlap_audio]) if len(overlap_audio) > 0 else deque()
            buffer_len = len(overlap_audio)

            chunk_idx += 1
            elapsed = time.time() - start_time
            chunk_audio = audio_np[:chunk_samples].copy()

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as f:
                sf.write(f.name, chunk_audio, SAMPLE_RATE)
                segments_gen, info = model.transcribe(
                    f.name,
                    language=language,
                    beam_size=5,
                    vad_filter=True,
                    vad_parameters=dict(
                        min_silence_duration_ms=200,  # short chunk: don't wait for long silence
                        speech_pad_ms=200,
                        threshold=0.3,
                    ),
                    word_timestamps=False,
                )
                segments = list(segments_gen)
                text = " ".join(seg.text.strip() for seg in segments)

            emit(f"[chunk {chunk_idx:03d} | {elapsed:6.1f}s | {info.language}] {text if text.strip() else '(silence)'}")

            if debug_log is not None:
                debug_log.append(dict(
                    chunk_idx=chunk_idx,
                    start_s=elapsed - chunk_s,
                    audio=chunk_audio,
                    text=text,
                    language=info.language,
                    language_probability=info.language_probability,
                    vad_segments=[(seg.start, seg.end) for seg in segments],
                ))

    t = threading.Thread(target=transcription_loop, daemon=True)
    t.start()

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=CHANNELS,
        dtype=DTYPE,
        blocksize=int(0.1 * SAMPLE_RATE),
        device=device_idx,
        callback=lambda indata, frames, time_info, status: _audio_callback(
            indata, frames, time_info, status, audio_queue
        ),
    ):
        while not stop_event.is_set():
            time.sleep(0.1)

    t.join(timeout=5)

In [6]:
# import sounddevice as sd

# print(sd.query_devices())

# DEFAULT_MIC_NAME = "MacBook Pro Microphone"

# def _find_device_by_name(name: str) -> int:
#     for i, d in enumerate(sd.query_devices()):
#         if d["name"] == name and d["max_input_channels"] > 0:
#             return i
#     print(f"[mic] '{name}' not found, falling back to system default")
#     return sd.default.device[0]

# MIC_DEVICE = _find_device_by_name(DEFAULT_MIC_NAME)

# debug_log = []
# stop_event = threading.Event()
# try:
#     stream_transcribe(model=model, stop_event=stop_event, chunk_s=3.0, device=MIC_DEVICE, debug_log=debug_log)
# except KeyboardInterrupt:
#     stop_event.set()

In [7]:
import io

normalizer = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords(),
])

results = []

for i, sample in enumerate(fleurs_id_samples):
    audio = sample["audio"]
    reference = sample["transcription"]

    audio_np, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as f:
        sf.write(f.name, audio_np, sr)
        result = transcribe(model, f.name, language="id")

    wer = jiwer.wer(reference, result.text, reference_transform=normalizer, hypothesis_transform=normalizer)
    results.append(dict(
        idx=i,
        reference=reference,
        hypothesis=result.text,
        wer=wer,
        rtf=result.realtime_factor,
    ))
    print(f"[{i:02d}] WER={wer:.3f} RTF={result.realtime_factor:.2f}")
    print(f"     ref: {reference}")
    print(f"     hyp: {result.text}")

avg_wer = sum(r["wer"] for r in results) / len(results)
avg_rtf = sum(r["rtf"] for r in results) / len(results)
print(f"\naverage WER: {avg_wer:.3f} | average RTF: {avg_rtf:.2f}")

[00] WER=0.375 RTF=3.46
     ref: ini adalah cep kelima martelly dalam empat tahun
     hyp: Ini adalah CEP ke-5 Marteli dalam 4 tahun.
[01] WER=0.071 RTF=3.13
     ref: anda dapat melihat piramidanya dalam gelap dan anda dapat melihatnya diam-diam sebelum pertunjukan mulai
     hyp: Anda dapat melihat piramidanya dalam gelap, dan Anda dapat melihatnya diam-diam sebelum pertunjukan dimulai.
[02] WER=0.000 RTF=2.78
     ref: walau ada tiga orang di dalam rumah yang ditabrak mobil tak satu pun yang cedera
     hyp: Walau ada tiga orang di dalam rumah yang ditabrak mobil, tak satu pun yang cedera.
[03] WER=0.077 RTF=3.70
     ref: sebagian besar distrik dilayani oleh bus coaster jepang kecil yang nyaman dan kokoh
     hyp: Sebagian besar distrik dilayani oleh bus kuaster Jepang kecil yang nyaman dan kokoh.
[04] WER=0.087 RTF=5.67
     ref: dari 1.400 orang yang disurvei sebelum pemilihan federal tahun 2010 mereka yang menentang australia menjadi republik tumbuh sebesar 8 persen sejak tahu